# 联邦学习网络入侵检测 - 实验复现

复现两篇硕士论文的实验结果：
1. **李可欣论文** - FedPCNN (FedProx + CNN-SVM) 和 分段式联邦学习 (LSTM + 阈值筛选)
2. **叶子超论文** - FedProx-CNN (FedProx + CNN + Stacking)

## 目标指标

### 第四章 FedPCNN 二分类
| 数据集 | Acc | Precision | FAR | Recall | F1 |
|--------|-----|-----------|-----|--------|----|
| NSL-KDD | 99.35% | 94.26% | 0.62% | 97.13% | 95.67% |
| UNSW-NB15 | 99.04% | 93.17% | 1.87% | 97.92% | 95.49% |
| CIC-IDS2017 | 99.79% | 98.91% | 0.58% | 99.63% | 99.27% |

### 第五章 分段式FL Non-IID
| 数据集 | Acc | FAR | F1 |
|--------|-----|-----|----|
| NSL-KDD | 98.65% | 1.79% | 95.10% |
| UNSW-NB15 | 97.83% | 3.06% | 94.55% |
| CIC-IDS2017 | 97.34% | 1.62% | 96.12% |

## 1. 环境安装

In [ ]:
# 检测运行环境
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals().get('get_ipython', lambda: None)())

if IN_COLAB:
    # 克隆仓库
    !git clone -b optimize-v1 https://github.com/george231224/FL.git 2>/dev/null || true
    %cd FL-optimize
else:
    # 本地运行 - 假设已在项目目录
    %cd /root/FL-optimize

# 安装依赖
!pip install -q -r requirements.txt

# 验证 GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 数据集下载

In [ ]:
import os
import requests
import zipfile
import io
import pandas as pd
import numpy as np

DATA_DIR = './data'

def download_file(url, dest_path):
    """Download a file with progress."""
    if os.path.exists(dest_path):
        print(f"  已存在: {dest_path}")
        return
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    print(f"  下载中: {url}")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"  已保存: {dest_path}")

# ── NSL-KDD ──
print("=" * 50)
print("下载 NSL-KDD")
print("=" * 50)
nsl_dir = f'{DATA_DIR}/NSL-KDD'
os.makedirs(nsl_dir, exist_ok=True)
nsl_base = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master'
download_file(f'{nsl_base}/KDDTrain%2B.txt', f'{nsl_dir}/KDDTrain+.txt')
download_file(f'{nsl_base}/KDDTest%2B.txt', f'{nsl_dir}/KDDTest+.txt')

# ── UNSW-NB15 ──
print("\n" + "=" * 50)
print("下载 UNSW-NB15")
print("=" * 50)
unsw_dir = f'{DATA_DIR}/UNSW_NB15'
os.makedirs(unsw_dir, exist_ok=True)
unsw_base = 'https://raw.githubusercontent.com/shramos/Adrehash/master/datasets/UNSW-NB15'
for fname in ['UNSW_NB15_training-set.csv', 'UNSW_NB15_testing-set.csv']:
    download_file(f'{unsw_base}/{fname}', f'{unsw_dir}/{fname}')

# ── CIC-IDS2017 ──
print("\n" + "=" * 50)
print("下载/准备 CIC-IDS2017")
print("=" * 50)
cicids_dir = f'{DATA_DIR}/CIC-IDS2017'
cicids_csv = f'{cicids_dir}/cicids2017_cleaned.csv'

if os.path.exists(cicids_csv):
    print(f"  已存在: {cicids_csv}")
else:
    os.makedirs(cicids_dir, exist_ok=True)
    # Try multiple sources for CIC-IDS2017
    # Option 1: Pre-cleaned version from common mirrors
    cicids_urls = [
        'https://raw.githubusercontent.com/ahlashkari/CICFlowMeter/master/ReadMe/GeneratedLabelledFlows.zip',
    ]
    
    print("  CIC-IDS2017 数据集较大，需要手动准备:")
    print("  1. 从 https://www.unb.ca/cic/datasets/ids-2017.html 下载")
    print("  2. 或使用 Kaggle: https://www.kaggle.com/datasets/cicdataset/cicids2017")
    print("  3. 合并所有 CSV 文件为 cicids2017_cleaned.csv")
    print("  4. 放入 ./data/CIC-IDS2017/ 目录")
    print("")
    print("  自动生成合成数据用于测试流程...")
    
    # Generate synthetic placeholder data for testing the pipeline
    np.random.seed(42)
    n_samples = 300000
    n_features = 52
    
    # Class distribution roughly matching CIC-IDS2017
    attack_types = ['Normal Traffic', 'DoS', 'DDoS', 'Port Scanning', 'Brute Force', 'Web Attacks', 'Bots']
    proportions = [0.55, 0.20, 0.10, 0.07, 0.04, 0.025, 0.015]
    labels = np.random.choice(attack_types, size=n_samples, p=proportions)
    
    features = np.random.randn(n_samples, n_features)
    # Add some class-specific signal
    for i, at in enumerate(attack_types):
        mask = labels == at
        features[mask, :5] += i * 0.5
    
    cols = [f'feature_{i}' for i in range(n_features)] + ['Attack Type']
    data = np.column_stack([features, labels])
    df = pd.DataFrame(data, columns=cols)
    for col in cols[:-1]:
        df[col] = df[col].astype(float)
    df.to_csv(cicids_csv, index=False)
    print(f"  合成数据已生成: {cicids_csv} ({n_samples} samples)")
    print("  注意: 使用真实数据集才能复现论文指标！")

print("\n数据集准备完成!")

## 3. 运行实验

### 3.1 FedPCNN 二分类（第四章）

In [ ]:
import subprocess
import sys
import json
import os

# 存储所有实验结果
all_results = {}

def run_experiment(cmd, label):
    """Run an experiment and capture output."""
    print(f"\n{'='*60}")
    print(f"实验: {label}")
    print(f"命令: {cmd}")
    print(f"{'='*60}")
    result = subprocess.run(
        cmd, shell=True,
        stdout=sys.stdout, stderr=sys.stderr,
        cwd=os.getcwd()
    )
    if result.returncode != 0:
        print(f"⚠️ 实验 {label} 返回码: {result.returncode}")
    return result.returncode

In [ ]:
# ── 3.1.1 NSL-KDD 二分类 (最小数据集，先验证) ──
run_experiment(
    'python main.py --model fedpcnn --dataset NSL-KDD --partition iid --classification binary --global-rounds 60',
    'FedPCNN NSL-KDD Binary IID'
)

In [ ]:
# ── 3.1.2 UNSW-NB15 二分类 ──
run_experiment(
    'python main.py --model fedpcnn --dataset UNSW-NB15 --partition iid --classification binary --global-rounds 60',
    'FedPCNN UNSW-NB15 Binary IID'
)

In [ ]:
# ── 3.1.3 CIC-IDS2017 二分类 ──
run_experiment(
    'python main.py --model fedpcnn --dataset CIC-IDS2017 --partition iid --classification binary --global-rounds 60',
    'FedPCNN CIC-IDS2017 Binary IID'
)

### 3.2 分段式联邦学习 Non-IID（第五章）

In [ ]:
# ── 3.2.1 NSL-KDD Non-IID 二分类 ──
run_experiment(
    'python main.py --model segmented --dataset NSL-KDD --partition non-iid --alpha 0.5 --classification binary --global-rounds 60',
    'Segmented FL NSL-KDD Binary Non-IID'
)

In [ ]:
# ── 3.2.2 UNSW-NB15 Non-IID 二分类 ──
run_experiment(
    'python main.py --model segmented --dataset UNSW-NB15 --partition non-iid --alpha 0.5 --classification binary --global-rounds 60',
    'Segmented FL UNSW-NB15 Binary Non-IID'
)

In [ ]:
# ── 3.2.3 CIC-IDS2017 Non-IID 二分类 ──
run_experiment(
    'python main.py --model segmented --dataset CIC-IDS2017 --partition non-iid --alpha 0.5 --classification binary --global-rounds 60',
    'Segmented FL CIC-IDS2017 Binary Non-IID'
)

### 3.3 补充实验：FedPCNN IID 多分类 + Segmented IID

In [ ]:
# FedPCNN 多分类 IID (NSL-KDD)
run_experiment(
    'python main.py --model fedpcnn --dataset NSL-KDD --partition iid --classification multi --global-rounds 60',
    'FedPCNN NSL-KDD Multi IID'
)

In [ ]:
# Segmented FL IID 二分类 (NSL-KDD)
run_experiment(
    'python main.py --model segmented --dataset NSL-KDD --partition iid --classification binary --global-rounds 60',
    'Segmented FL NSL-KDD Binary IID'
)

## 4. 结果汇总与论文指标对比

In [ ]:
import json
import os
import glob
import pandas as pd
from IPython.display import display, HTML

# ── 论文目标指标 ──
paper_ch4 = {
    'NSL-KDD':     {'Accuracy': 99.35, 'Precision': 94.26, 'FAR': 0.62, 'Recall': 97.13, 'F1-Score': 95.67},
    'UNSW-NB15':   {'Accuracy': 99.04, 'Precision': 93.17, 'FAR': 1.87, 'Recall': 97.92, 'F1-Score': 95.49},
    'CIC-IDS2017': {'Accuracy': 99.79, 'Precision': 98.91, 'FAR': 0.58, 'Recall': 99.63, 'F1-Score': 99.27},
}

paper_ch5 = {
    'NSL-KDD':     {'Accuracy': 98.65, 'FAR': 1.79, 'F1-Score': 95.10},
    'UNSW-NB15':   {'Accuracy': 97.83, 'FAR': 3.06, 'F1-Score': 94.55},
    'CIC-IDS2017': {'Accuracy': 97.34, 'FAR': 1.62, 'F1-Score': 96.12},
}

# ── 加载实验结果 ──
results_dir = './results'
result_files = glob.glob(f'{results_dir}/*.json')

exp_results = {}
for fpath in sorted(result_files):
    with open(fpath) as f:
        data = json.load(f)
    key = f"{data['model']}_{data['dataset']}_{data['partition']}_{data.get('classification', 'multi')}"
    exp_results[key] = data

print(f"加载了 {len(exp_results)} 个实验结果")
for k in exp_results:
    print(f"  {k}")

In [ ]:
# ── 第四章 FedPCNN 二分类对比表 ──
print("\n" + "=" * 80)
print("第四章 FedPCNN 二分类 - 实验结果 vs 论文目标")
print("=" * 80)

rows = []
for ds in ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017']:
    key = f"fedpcnn_{ds}_iid_binary"
    if key in exp_results:
        m = exp_results[key]['metrics']
        paper = paper_ch4[ds]
        for metric in ['Accuracy', 'Precision', 'FAR', 'Recall', 'F1-Score']:
            exp_val = m.get(metric, 0)
            paper_val = paper[metric]
            diff = exp_val - paper_val
            rows.append({
                'Dataset': ds,
                'Metric': metric,
                'Paper': f'{paper_val:.2f}%',
                'Ours': f'{exp_val:.2f}%',
                'Diff': f'{diff:+.2f}%',
                'Status': '✅' if abs(diff) < 2.0 else ('⚠️' if abs(diff) < 5.0 else '❌')
            })
    else:
        rows.append({'Dataset': ds, 'Metric': '-', 'Paper': '-', 'Ours': '未运行', 'Diff': '-', 'Status': '⏳'})

df_ch4 = pd.DataFrame(rows)
display(df_ch4)

In [ ]:
# ── 第五章 分段式FL Non-IID 对比表 ──
print("\n" + "=" * 80)
print("第五章 分段式FL Non-IID - 实验结果 vs 论文目标")
print("=" * 80)

rows = []
for ds in ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017']:
    key = f"segmented_{ds}_non-iid_binary"
    if key in exp_results:
        m = exp_results[key]['metrics']
        paper = paper_ch5[ds]
        for metric in ['Accuracy', 'FAR', 'F1-Score']:
            exp_val = m.get(metric, 0)
            paper_val = paper[metric]
            diff = exp_val - paper_val
            rows.append({
                'Dataset': ds,
                'Metric': metric,
                'Paper': f'{paper_val:.2f}%',
                'Ours': f'{exp_val:.2f}%',
                'Diff': f'{diff:+.2f}%',
                'Status': '✅' if abs(diff) < 2.0 else ('⚠️' if abs(diff) < 5.0 else '❌')
            })
    else:
        rows.append({'Dataset': ds, 'Metric': '-', 'Paper': '-', 'Ours': '未运行', 'Diff': '-', 'Status': '⏳'})

df_ch5 = pd.DataFrame(rows)
display(df_ch5)

## 5. 可视化

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# 显示所有已生成的图表
plot_dir = './results/plots'
if os.path.exists(plot_dir):
    plot_files = sorted(glob.glob(f'{plot_dir}/*.png'))
    print(f"找到 {len(plot_files)} 张图表")
    
    for pf in plot_files:
        fig, ax = plt.subplots(figsize=(12, 8))
        img = mpimg.imread(pf)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(os.path.basename(pf), fontsize=12)
        plt.tight_layout()
        plt.show()
else:
    print("未找到图表目录，请先运行实验")

In [ ]:
# ── 手动绘制混淆矩阵（如果实验已运行但图表缺失）──
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_cm_from_results(result_key, title='Confusion Matrix'):
    """从 JSON 结果中重建混淆矩阵（需要保存 y_true/y_pred）"""
    # 目前 JSON 中不保存 y_true/y_pred，使用已保存的图片
    plot_path = f'./results/plots/{result_key}_cm.png'
    if os.path.exists(plot_path):
        fig, ax = plt.subplots(figsize=(10, 8))
        img = mpimg.imread(plot_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title)
        plt.show()
    else:
        print(f"未找到: {plot_path}")

# 显示 FedPCNN 二分类混淆矩阵
for ds in ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017']:
    plot_cm_from_results(f'FedPCNN_{ds}_iid_binary', f'FedPCNN {ds} Binary Confusion Matrix')

In [ ]:
# ── 损失曲线对比 ──
for ds in ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017']:
    loss_path = f'./results/plots/FedPCNN_{ds}_iid_binary_loss.png'
    if os.path.exists(loss_path):
        fig, ax = plt.subplots(figsize=(10, 6))
        img = mpimg.imread(loss_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'FedPCNN {ds} Training Loss')
        plt.show()

## 6. 综合对比图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── 综合对比: 论文 vs 实验 ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ch4: FedPCNN Binary
ax = axes[0]
datasets = ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017']
metrics_list = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
x = np.arange(len(datasets))
width = 0.15

for i, metric in enumerate(metrics_list):
    paper_vals = [paper_ch4[ds][metric] for ds in datasets]
    exp_vals = []
    for ds in datasets:
        key = f"fedpcnn_{ds}_iid_binary"
        if key in exp_results:
            exp_vals.append(exp_results[key]['metrics'].get(metric, 0))
        else:
            exp_vals.append(0)
    
    ax.bar(x + i*width - width*1.5, paper_vals, width, label=f'Paper {metric}', alpha=0.6)
    ax.bar(x + i*width - width*1.5 + width*4, exp_vals, width, label=f'Ours {metric}', alpha=0.9)

ax.set_xlabel('Dataset')
ax.set_ylabel('%')
ax.set_title('Ch4: FedPCNN Binary - Paper vs Ours')
ax.set_xticks(x + width)
ax.set_xticklabels(datasets)
ax.set_ylim(90, 101)
ax.legend(fontsize=7, ncol=2)
ax.grid(axis='y', alpha=0.3)

# Ch5: Segmented FL Non-IID
ax = axes[1]
metrics_list_ch5 = ['Accuracy', 'F1-Score']

for i, metric in enumerate(metrics_list_ch5):
    paper_vals = [paper_ch5[ds][metric] for ds in datasets]
    exp_vals = []
    for ds in datasets:
        key = f"segmented_{ds}_non-iid_binary"
        if key in exp_results:
            exp_vals.append(exp_results[key]['metrics'].get(metric, 0))
        else:
            exp_vals.append(0)
    
    offset = i * width
    ax.bar(x + offset - width/2, paper_vals, width, label=f'Paper {metric}', alpha=0.6, color=f'C{i}')
    ax.bar(x + offset + width*1.5, exp_vals, width, label=f'Ours {metric}', alpha=0.9, color=f'C{i+2}')

ax.set_xlabel('Dataset')
ax.set_ylabel('%')
ax.set_title('Ch5: Segmented FL Non-IID Binary - Paper vs Ours')
ax.set_xticks(x + width/2)
ax.set_xticklabels(datasets)
ax.set_ylim(90, 101)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./results/plots/paper_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("综合对比图已保存: ./results/plots/paper_comparison.png")

## 7. Summary CSV

In [ ]:
summary_csv = './results/summary.csv'
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    display(df)
else:
    print("未找到 summary.csv，请先运行实验")